In [ ]:
"""
train_photo_detector.py

Script complet pour entraîner un petit CNN (classification binaire photo vs non-photo).
Structure attendue:
 data/
   photos/       <-- images considérées comme "photo"
   non-photos/   <-- images non-photo (peintures, dessins, schémas, texte...)

Usage: python train_photo_detector.py
"""

import os
from pathlib import Path
import math
import json
import datetime

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers, losses, metrics
import numpy as np

# -----------------------------
# === Paramètres principaux ===
# -----------------------------
# Ces paramètres peuvent être modifiés avant de lancer l'entraînement.

CONFIG = {
    # Chemin vers le dossier parent contenant deux sous-dossiers 'photos' et 'non-photos'
    "data_dir": "data",  # <-- chemin attendu : data/photos et data/non-photos

    # Image input
    "img_size": (224, 224),   # Taille to which images will be resized (height, width)
    "channels": 3,            # 3 = RGB, 1 = grayscale

    # Training
    "batch_size": 32,         # taille de batch (ajuster selon GPU/CPU)
    "epochs": 100,            # nombre total d'époques (le temps n'est pas contraint selon toi)
    "seed": 42,               # graine pour reproductibilité

    # Model capacity (petit modèle) :
    # filters_multiplier : 0.75 -> plus petit, 1.0 -> default, 1.25 -> plus grand
    "filters_multiplier": 1.0,

    # Dropout head
    "dropout_head": 0.3,

    # SE blocks ratio (0 = désactivé, >0 = squeeze-excite léger)
    "se_ratio": 0.125,

    # Optimizer & LR
    "learning_rate": 1e-3,

    # Callbacks / sauvegardes
    "save_dir": "checkpoints",   # dossier de sauvegarde des modèles et logs
    "save_best_name": "best_model.h5",
    "periodic_save_every_n_epochs": 5,  # sauvegarde du modèle complet tous les N epochs
    "save_periodic_template": "model_epoch_{epoch:03d}.h5",

    # Augmentation
    "use_augmentation": True,

    # MixUp (simple implementation) - si actif, mélange images/labels
    "use_mixup": False,
    "mixup_alpha": 0.2,

    # Label mapping : si 'photos' doit être 1 (positif), sinon inverse
    "positive_class_name": "photos",  # nom du dossier qui correspond à la classe positive (label=1)
}

# Create save dir
os.makedirs(CONFIG["save_dir"], exist_ok=True)


# -----------------------------
# === Utilitaires & modèle ===
# -----------------------------
def swish(x):
    return tf.nn.swish(x)

def conv_block(x, filters, kernel=3, strides=1, se_ratio=0.0, drop_rate=0.0):
    """Depthwise separable conv -> BN -> swish (option SE + dropout)."""
    x = layers.SeparableConv2D(filters, kernel, padding='same', strides=strides, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(swish)(x)
    if se_ratio and se_ratio > 0:
        se = layers.GlobalAveragePooling2D()(x)
        se = layers.Reshape((1,1,filters))(se)
        # small bottleneck
        ch = max(1, int(filters * se_ratio))
        se = layers.Conv2D(ch, 1, activation=swish, padding='same')(se)
        se = layers.Conv2D(filters, 1, activation='sigmoid', padding='same')(se)
        x = layers.Multiply()([x, se])
    if drop_rate and drop_rate > 0:
        x = layers.Dropout(drop_rate)(x)
    return x

def residual_down(x, filters, stride, se_ratio=0.0):
    """Residual block composed of two separable convs with a shortcut (strided)."""
    shortcut = layers.SeparableConv2D(filters, 1, strides=stride, padding='same', use_bias=False)(x)
    shortcut = layers.BatchNormalization()(shortcut)
    x = conv_block(x, filters, strides=stride, se_ratio=se_ratio)
    x = conv_block(x, filters, strides=1, se_ratio=se_ratio)
    x = layers.Add()([x, shortcut])
    return x

def build_compact_cnn(input_shape=(224,224,3), filters_multiplier=1.0, dropout_head=0.2, se_ratio=0.0):
    """
    Construit le modèle compact.
    - input_shape : (H, W, C)
    - filters_multiplier : scale des filtres pour augmenter/diminuer la taille du modèle
    - dropout_head : dropout avant la tête finale
    - se_ratio : squeeze-and-excite ratio
    """
    def F(n): return max(8, int(n * filters_multiplier))
    inp = layers.Input(shape=input_shape, name="input_image")

    # Optional normalization layer (rescale to [0,1])
    x = layers.Rescaling(1.0 / 255.0)(inp)

    # Stem
    x = layers.Conv2D(F(16), 3, strides=2, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(swish)(x)

    # Stages : (filters, repeats, stride)
    stages = [
        (24, 1, 1),
        (40, 2, 2),
        (80, 3, 2),
        (160, 3, 2),
    ]
    for filters, repeats, stride in stages:
        for i in range(repeats):
            s = stride if i == 0 else 1
            x = residual_down(x, F(filters), stride=s, se_ratio=se_ratio)

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(F(128), use_bias=False, name="proj")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation(swish)(x)
    x = layers.Dropout(dropout_head)(x)
    out = layers.Dense(1, activation='sigmoid', name="pred")(x)

    model = models.Model(inputs=inp, outputs=out, name="compact_photo_cnn")
    return model


# -----------------------------
# === Dataset pipeline utils ===
# -----------------------------
def count_files_in_dir(dirpath):
    p = Path(dirpath)
    if not p.exists():
        return 0
    return sum(1 for _ in p.rglob("*") if _.is_file())

def prepare_datasets(data_dir, img_size, batch_size, seed=42, val_split=0.15, test_split=0.05):
    """
    Charge les données depuis data_dir en utilisant image_dataset_from_directory.
    On crée train/val/test splits.
    Structure attendue :
      data_dir/
         photos/
         non-photos/
    Retourne: train_ds, val_ds, test_ds, class_names
    """
    # We first create a dataset with 1 - (val+test) for training, etc.
    total_val = val_split + test_split
    train_ds = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        labels='inferred',
        label_mode='binary',
        class_names=None,  # keeps alphabetical order of folder names
        color_mode="rgb" if CONFIG["channels"] == 3 else "grayscale",
        batch_size=batch_size,
        image_size=img_size,
        shuffle=True,
        seed=seed,
        validation_split=total_val,
        subset="training"
    )
    # We'll create a combined val+test and then split it locally
    valtest_ds = tf.keras.preprocessing.image_dataset_from_directory(
        data_dir,
        labels='inferred',
        label_mode='binary',
        class_names=None,
        color_mode="rgb" if CONFIG["channels"] == 3 else "grayscale",
        batch_size=batch_size,
        image_size=img_size,
        shuffle=True,
        seed=seed,
        validation_split=total_val,
        subset="validation"
    )
    # Determine sizes to split valtest into validation and test
    # Note: keras generator returns batches; we compute cardinality
    valtest_count = 0
    for batch in valtest_ds:
        valtest_count += batch[0].shape[0]
    # compute counts
    val_count = int(round(val_split / total_val * valtest_count))
    # Build val and test by unbatching then re-batching (deterministic split)
    vt_unbatched = valtest_ds.unbatch()
    val_ds = vt_unbatched.take(val_count).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    test_ds = vt_unbatched.skip(val_count).batch(batch_size).prefetch(tf.data.AUTOTUNE)

    # Ensure train prefetch/cache happens after augmentation stage in training pipeline
    class_names = train_ds.class_names  # order alphabetical
    return train_ds, val_ds, test_ds, class_names

def build_augmentation_layer(img_size):
    """Keras preprocessing layers for augmentation (applied on the fly)."""
    data_aug = tf.keras.Sequential(name="data_augmentation", layers=[
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomZoom(0.12),
        layers.RandomContrast(0.12),
        # brightness/gaussian noise etc can be added
    ])
    return data_aug

def mixup_batch(images, labels, alpha=0.2):
    """Simple MixUp implementation for a batch of images & binary labels."""
    if alpha <= 0:
        return images, labels
    batch = tf.shape(images)[0]
    beta = tf.random.gamma(shape=[batch], alpha=alpha)
    # Draw lambda from Beta distribution (symmetric)
    lam = tf.random.uniform([batch], 0, 1)
    lam = tf.cast(lam, images.dtype)
    # For simplicity use permutation
    indices = tf.random.shuffle(tf.range(batch))
    mixed_images = images * tf.reshape(lam, (-1,1,1,1)) + tf.gather(images, indices) * tf.reshape(1.0 - lam, (-1,1,1,1))
    mixed_labels = labels * tf.reshape(lam, (-1,1)) + tf.gather(labels, indices) * tf.reshape(1.0 - lam, (-1,1))
    return mixed_images, mixed_labels

# -----------------------------
# === Custom callbacks ===
# -----------------------------
class PeriodicSaver(callbacks.Callback):
    """Sauvegarde le modèle complet (format h5) tous les N epochs."""
    def __init__(self, folder, template, every_n_epochs=5):
        super().__init__()
        self.folder = folder
        self.template = template
        self.every = every_n_epochs
        os.makedirs(folder, exist_ok=True)

    def on_epoch_end(self, epoch, logs=None):
        ep = epoch + 1
        if self.every > 0 and (ep % self.every == 0):
            fname = os.path.join(self.folder, self.template.format(epoch=ep))
            # Save model weights + optimizer state
            self.model.save(fname)
            print(f"\n[PeriodicSaver] modèle sauvegardé : {fname}")

# -----------------------------
# === Main routine ===
# -----------------------------
def main(cfg):
    tf.random.set_seed(cfg["seed"])
    np.random.seed(cfg["seed"])

    data_dir = cfg["data_dir"]
    if not os.path.isdir(data_dir):
        raise RuntimeError(f"Data directory {data_dir} not found. Créez {data_dir}/photos et {data_dir}/non-photos.")

    # Prepare datasets
    print("==> Préparation des datasets (train / val / test)...")
    train_ds_raw, val_ds, test_ds, class_names = prepare_datasets(
        data_dir,
        img_size=cfg["img_size"],
        batch_size=cfg["batch_size"],
        seed=cfg["seed"]
    )
    print("Classes trouvées (ordre alphabétique):", class_names)

    # We want the positive class label index according to the folder name
    try:
        positive_index = class_names.index(cfg["positive_class_name"])
    except ValueError:
        # if not found, default: index 0 is positive (rare), but we warn
        print(f"Warning: '{cfg['positive_class_name']}' not found in {class_names}. On utilisera l'indice 0 comme positif.")
        positive_index = 0

    # Count files to compute class weights
    photos_count = count_files_in_dir(os.path.join(data_dir, "photos"))
    nonphotos_count = count_files_in_dir(os.path.join(data_dir, "non-photos"))
    total_count = photos_count + nonphotos_count
    if total_count == 0:
        raise RuntimeError("Aucune image trouvée dans data/photos ou data/non-photos.")
    print(f"Images: photos={photos_count}, non-photos={nonphotos_count}, total={total_count}")

    # class weights : inverse proportion to frequency
    # Keras expects a dict mapping label_index -> weight
    # find actual index of 'photos' and 'non-photos' in class_names
    # class_names is alphabetical order => might be ['non-photos', 'photos'] or reversed
    mapping = {}
    # build mapping from folder name to class index
    folder_to_index = {name: idx for idx, name in enumerate(class_names)}
    # compute weights (safe fallback if counts=0)
    weight_photos = (total_count / photos_count) if photos_count > 0 else 1.0
    weight_nonphotos = (total_count / nonphotos_count) if nonphotos_count > 0 else 1.0
    # assign
    if "photos" in folder_to_index:
        mapping[folder_to_index["photos"]] = weight_photos
    if "non-photos" in folder_to_index:
        mapping[folder_to_index["non-photos"]] = weight_nonphotos

    print("Class weights utilisées:", mapping)

    # Build augmentation layer (applied to training only)
    augmentation_layer = build_augmentation_layer(cfg["img_size"]) if cfg["use_augmentation"] else None

    # Prepare final datasets with augmentation & prefetch
    def preprocess_train(images, labels):
        # images: uint8 [0..255], labels: scalar (0 or 1)
        if augmentation_layer is not None:
            images = augmentation_layer(images)
        # convert labels shape (batch,) -> (batch,1) for mixup compatibility
        labels = tf.expand_dims(labels, axis=-1)
        return images, labels

    train_ds = train_ds_raw.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
    val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
    test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

    # Build model
    print("==> Construction du modèle ...")
    input_shape = (cfg["img_size"][0], cfg["img_size"][1], cfg["channels"])
    model = build_compact_cnn(input_shape=input_shape,
                              filters_multiplier=cfg["filters_multiplier"],
                              dropout_head=cfg["dropout_head"],
                              se_ratio=cfg["se_ratio"])
    # Print and save model architecture
    model.summary()
    summary_txt = os.path.join(cfg["save_dir"], "model_architecture.txt")
    with open(summary_txt, "w") as f:
        model.summary(print_fn=lambda s: f.write(s + "\n"))
    print(f"Architecture saved to {summary_txt}")

    # Compile
    opt = optimizers.Adam(learning_rate=cfg["learning_rate"])
    loss = losses.BinaryCrossentropy()
    model.compile(optimizer=opt,
                  loss=loss,
                  metrics=[metrics.AUC(name="auc"), metrics.BinaryAccuracy(name="acc"),
                           metrics.Precision(name="precision"), metrics.Recall(name="recall")])

    # Callbacks
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    csvlog = callbacks.CSVLogger(os.path.join(cfg["save_dir"], f"training_log_{timestamp}.csv"))
    # save best model on val loss (or val_auc if you prefer)
    best_path = os.path.join(cfg["save_dir"], cfg["save_best_name"])
    cp_best = callbacks.ModelCheckpoint(best_path, monitor="val_loss", save_best_only=True, save_weights_only=False)
    periodic = PeriodicSaver(cfg["save_dir"], cfg["save_periodic_template"], every_n_epochs=cfg["periodic_save_every_n_epochs"])
    reduce_lr = callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-7, verbose=1)
    earlystop = callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True)

    cb_list = [csvlog, cp_best, periodic, reduce_lr, earlystop]

    # Option: MixUp augmentation per-batch inside training loop -> use dataset.map to apply mixup
    if cfg["use_mixup"]:
        def mixup_map(images, labels):
            mixed_images, mixed_labels = mixup_batch(images, labels, alpha=cfg["mixup_alpha"])
            return mixed_images, mixed_labels
        train_ds = train_ds.map(mixup_map, num_parallel_calls=tf.data.AUTOTUNE)

    # Fit
    print("==> Démarrage de l'entraînement ...")
    # compute steps per epoch (approx)
    steps_per_epoch = None
    try:
        # if dataset supports cardinality
        card = tf.data.experimental.cardinality(train_ds).numpy()
        if card > 0:
            steps_per_epoch = card
    except Exception:
        steps_per_epoch = None

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=cfg["epochs"],
        callbacks=cb_list,
        class_weight=mapping,
        verbose=1
    )

    # Evaluate on test set
    print("==> Évaluation finale sur test set ...")
    res = model.evaluate(test_ds)
    print("Test results:", res)

    # Save final model
    final_path = os.path.join(cfg["save_dir"], f"final_model_{timestamp}.h5")
    model.save(final_path)
    print(f"Modèle final sauvegardé : {final_path}")

    # Export config + history
    with open(os.path.join(cfg["save_dir"], f"config_{timestamp}.json"), "w") as f:
        json.dump(cfg, f, indent=2)
    print("Configuration sauvegardée.")

if __name__ == "__main__":
    main(CONFIG)
